In [1]:
#verify values from startkind
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/clean/trials.db")

q = """
SELECT startKind, COUNT(*) AS studies
FROM studies
GROUP BY startKind
"""
print(pd.read_sql(q, conn))

  startKind  studies
0    actual     1663
1   planned       45
2   unknown        9


In [8]:
#how many studies are conducted/year
q1 = """
SELECT CAST(startYear AS INTEGER) AS startYear, -- sets year as integer
       COUNT(*) AS studies
FROM studies
WHERE startKind = 'actual'
GROUP BY startYear
ORDER BY startYear
"""
print(pd.read_sql(q1, conn))

    startYear  studies
0        2015      154
1        2016      131
2        2017      136
3        2018      118
4        2019      121
5        2020      114
6        2021      128
7        2022      150
8        2023      140
9        2024      184
10       2025      201
11       2026       86


In [9]:
q = """
SELECT status, COUNT(*) AS studies
FROM studies
GROUP BY status
ORDER BY studies DESC
"""
print(pd.read_sql(q, conn))

                    status  studies
0                COMPLETED      795
1               RECRUITING      346
2    ACTIVE_NOT_RECRUITING      291
3               TERMINATED      167
4                  UNKNOWN       67
5       NOT_YET_RECRUITING       24
6  ENROLLING_BY_INVITATION       15
7                WITHDRAWN       10
8                SUSPENDED        2


In [10]:
# area x phases that domiate active studies
q2 = """
SELECT a.area,
       s.phaseLabel,
       COUNT(DISTINCT s.nctId) AS studies,
       SUM(CASE WHEN s.status = 'NOT_YET_RECRUITING' THEN 1 ELSE 0 END) AS notYet,
       COUNT(DISTINCT s.sponsor) AS sponsors
FROM studies AS s
JOIN study_area AS a ON a.nctId = s.nctId -- conencts rows through nctId
WHERE s.status IN ('RECRUITING', 'ACTIVE_NOT_RECRUITING',
                   'ENROLLING_BY_INVITATION', 'NOT_YET_RECRUITING')
GROUP BY a.area, s.phaseLabel
ORDER BY studies DESC
"""
print(pd.read_sql(q2, conn).head(25))

                          area phaseLabel  studies  notYet  sponsors
0                     Oncology    Phase 3      177       0        69
1                Immune system    Phase 3       87       0        43
2                  Respiratory    Phase 3       85       0        35
3   Skin and connective tissue    Phase 3       70       0        32
4          Blood and lymphatic    Phase 3       56       1        37
5                    Digestive    Phase 3       52       0        30
6               Cardiovascular    Phase 3       48       0        28
7      Metabolic and endocrine    Phase 3       45       3        26
8               Nervous system    Phase 3       41       1        22
9                   Urogenital    Phase 3       37       0        19
10              Not classified    Phase 3       34       1        25
11                 Respiratory    Phase 2       25       2        22
12                    Oncology    Phase 2       24       2        16
13              Not classified   N

In [12]:
#competing studies
print(pd.read_sql("""
SELECT COUNT(*) AS n FROM studies
WHERE status IN ('RECRUITING','ACTIVE_NOT_RECRUITING','ENROLLING_BY_INVITATION','NOT_YET_RECRUITING')
""", conn))

     n
0  676


In [19]:
# q2 + cities + sites
q3 = """
SELECT a.area,
       s.phaseLabel,
       COUNT(DISTINCT s.nctId) AS studies,
       COUNT(DISTINCT CASE WHEN s.status = 'NOT_YET_RECRUITING' THEN s.nctId END) AS notYet,
       COUNT(DISTINCT s.sponsor) AS sponsors,
       COUNT(DISTINCT CASE WHEN t.city <> 'Unknown city' THEN t.city END) AS cities,
       COUNT(DISTINCT CASE WHEN t.facility <> 'Generic site'
                           THEN t.city || '|' || t.facility END) AS namedCentres,
       CASE WHEN COUNT(DISTINCT s.nctId) < 5 THEN 'limited data' ELSE '' END AS note
FROM studies AS s
JOIN study_area AS a ON a.nctId = s.nctId
JOIN sites AS t ON t.nctId = s.nctId
WHERE s.status IN ('RECRUITING', 'ACTIVE_NOT_RECRUITING',
                   'ENROLLING_BY_INVITATION', 'NOT_YET_RECRUITING')
GROUP BY a.area, s.phaseLabel
ORDER BY studies DESC;
"""
print(pd.read_sql(q3, conn).head(25).to_string())

                          area phaseLabel  studies  notYet  sponsors  cities  namedCentres note
0                     Oncology    Phase 3      177       0        69      23           256     
1                Immune system    Phase 3       87       0        43      23           106     
2                  Respiratory    Phase 3       85       0        35      26           141     
3   Skin and connective tissue    Phase 3       70       0        32      21           135     
4          Blood and lymphatic    Phase 3       56       1        37      14            62     
5                    Digestive    Phase 3       52       0        30      18           139     
6               Cardiovascular    Phase 3       48       0        28      27           125     
7      Metabolic and endocrine    Phase 3       45       3        26      27           128     
8               Nervous system    Phase 3       41       1        22      26            88     
9                   Urogenital    Phase 

In [20]:
q = """
SELECT status, startKind, COUNT(*) AS studies
FROM studies
WHERE status IN ('TERMINATED', 'WITHDRAWN')
GROUP BY status, startKind
"""
print(pd.read_sql(q, conn))

       status startKind  studies
0  TERMINATED    actual      167
1   WITHDRAWN    actual        2
2   WITHDRAWN   planned        7
3   WITHDRAWN   unknown        1


In [23]:
#stop rate
q4 = """
SELECT phaseLabel,
       COUNT(*) AS studies,
       SUM(CASE WHEN status IN ('TERMINATED', 'WITHDRAWN') THEN 1 ELSE 0 END) AS stopped,
       ROUND(100.0 * SUM(CASE WHEN status IN ('TERMINATED', 'WITHDRAWN') THEN 1 ELSE 0 END)
             / COUNT(*), 1) AS stopPct
FROM studies
WHERE startYear BETWEEN 2015 AND 2021
  AND status IN ('COMPLETED', 'TERMINATED', 'WITHDRAWN')
GROUP BY phaseLabel
ORDER BY studies DESC
"""
print(pd.read_sql(q4, conn))

  phaseLabel  studies  stopped  stopPct
0    Phase 3      417       75     18.0
1    Phase 2      151       34     22.5
2   No phase       88        2      2.3
3    Phase 4       27        2      7.4
4    Phase 1       26        3     11.5
5  Phase 2/3       18       10     55.6
6  Phase 1/2       12        4     33.3


In [24]:
# in which cties do the most studies appear
q5 = """
SELECT city,
       COUNT(DISTINCT nctId) AS studies,
       ROUND(100.0 * COUNT(DISTINCT nctId) / (SELECT COUNT(*) FROM studies), 1) AS pctOfStudies
FROM sites
WHERE city <> 'Unknown city'
GROUP BY city
ORDER BY studies DESC
LIMIT 15
"""
print(pd.read_sql(q5, conn))

           city  studies  pctOfStudies
0     Bucharest     1201          69.9
1   Cluj-Napoca      799          46.5
2     Timisoara      607          35.4
3       Craiova      487          28.4
4          Iasi      388          22.6
5        Brasov      311          18.1
6   Targu Mures      244          14.2
7        Oradea      213          12.4
8     Constanta      168           9.8
9         Sibiu      121           7.0
10    Baia Mare      111           6.5
11     Floresti      104           6.1
12     Ploiesti       85           5.0
13      Suceava       83           4.8
14       Galati       78           4.5


In [25]:
#how many studies are conducted 1, 2, 3.. cities?
q = """
SELECT nCities, COUNT(*) AS studies
FROM (
    SELECT nctId, COUNT(DISTINCT city) AS nCities
    FROM sites
    WHERE city <> 'Unknown city'
    GROUP BY nctId
)
GROUP BY nCities
ORDER BY nCities
"""
print(pd.read_sql(q, conn))

    nCities  studies
0         1      541
1         2      290
2         3      264
3         4      202
4         5      140
5         6      116
6         7       58
7         8       31
8         9       19
9        10       17
10       11       17
11       12        6
12       13        4
13       14        3
14       15        4
15       16        1
16       18        1
17       19        1


In [27]:
# how many studies have at least one center in the top N cities, excluding duplicates
q6 = """
WITH city_rank AS (
    SELECT city,
           ROW_NUMBER() OVER (ORDER BY COUNT(DISTINCT nctId) DESC) AS rnk
    FROM sites
    WHERE city <> 'Unknown city'
    GROUP BY city
),
study_best AS (
    SELECT t.nctId, MIN(r.rnk) AS bestRank
    FROM sites AS t
    JOIN city_rank AS r ON r.city = t.city
    GROUP BY t.nctId
)
SELECT b.bestRank AS topN,
       c.city,
       COUNT(*) AS newStudies,
       SUM(COUNT(*)) OVER (ORDER BY b.bestRank) AS cumStudies,
       ROUND(100.0 * SUM(COUNT(*)) OVER (ORDER BY b.bestRank)
             / (SELECT COUNT(*) FROM studies), 1) AS cumPct
FROM study_best AS b
JOIN city_rank AS c ON c.rnk = b.bestRank
GROUP BY b.bestRank, c.city
ORDER BY b.bestRank
LIMIT 10
"""
print(pd.read_sql(q6, conn))

   topN         city  newStudies  cumStudies  cumPct
0     1    Bucharest        1201        1201    69.9
1     2  Cluj-Napoca         256        1457    84.9
2     3    Timisoara         104        1561    90.9
3     4      Craiova          25        1586    92.4
4     5         Iasi          32        1618    94.2
5     6       Brasov          16        1634    95.2
6     7  Targu Mures          19        1653    96.3
7     8       Oradea           5        1658    96.6
8     9    Constanta          12        1670    97.3
9    10        Sibiu           7        1677    97.7


In [34]:
#sponsors analysis based on class
q = """
SELECT sponsorClass, COUNT(*) AS studies, COUNT(DISTINCT sponsor) AS sponsors
FROM studies
GROUP BY sponsorClass
ORDER BY studies DESC
"""
print(pd.read_sql(q, conn))

  sponsorClass  studies  sponsors
0     INDUSTRY     1433       377
1        OTHER      265       106
2      NETWORK       13         3
3    OTHER_GOV        6         4


In [36]:
# one row per class - how many studies and how many distinc sponsors each has. other (universities, CRO, hospitals) and network (research networks, cooperative groups)
q = """
SELECT sponsorClass, sponsor, COUNT(*) AS studies
FROM studies
WHERE sponsorClass IN ('OTHER', 'NETWORK')
GROUP BY sponsorClass, sponsor
ORDER BY sponsorClass, studies DESC
LIMIT 30
"""
print(pd.read_sql(q, conn).to_string())

   sponsorClass                                                                                      sponsor  studies
0       NETWORK                                                    I.R.A. Istituto Ricerche Applicate S.p.A.       11
1       NETWORK                                            Iacob Czihac Emergency Military Clinical Hospital        1
2       NETWORK                                                                 Canadian Cancer Trials Group        1
3         OTHER                                                                      Babes-Bolyai University       41
4         OTHER                                          Iuliu Hatieganu University of Medicine and Pharmacy       27
5         OTHER                                          Grigore T. Popa University of Medicine and Pharmacy       13
6         OTHER                                                                 West University of Timisoara       12
7         OTHER                                         

In [40]:
#top 40 industry sponsors
q = """
SELECT sponsor, COUNT(*) AS studies
FROM studies
WHERE sponsorClass = 'INDUSTRY'
GROUP BY sponsor
ORDER BY studies DESC
LIMIT 40
"""
print(pd.read_sql(q, conn).to_string())

                                                        sponsor  studies
0                                          Bristol-Myers Squibb       93
1                                         Eli Lilly and Company       64
2                                       Merck Sharp & Dohme LLC       62
3                                      Novartis Pharmaceuticals       60
4                                                   AstraZeneca       59
5                                             Hoffmann-La Roche       45
6                                                         Amgen       44
7                                               GlaxoSmithKline       43
8                                                        Sanofi       40
9                                              Novo Nordisk A/S       37
10                                                       AbbVie       37
11                                         Boehringer Ingelheim       28
12                          Janssen Research & Deve

In [43]:
#sponsors for alias table
q = """
SELECT sponsor, COUNT(*) AS studies
FROM studies
WHERE sponsorClass = 'INDUSTRY'
  AND (sponsor LIKE '%bristol%' OR sponsor LIKE '%lilly%' OR sponsor LIKE '%merck%'
       OR sponsor LIKE '%msd%' OR sponsor LIKE '%novartis%' OR sponsor LIKE '%astrazeneca%'
       OR sponsor LIKE '%roche%' OR sponsor LIKE '%genentech%' OR sponsor LIKE '%amgen%'
       OR sponsor LIKE '%glaxo%' OR sponsor LIKE '%gsk%' OR sponsor LIKE '%sanofi%'
       OR sponsor LIKE '%novo%' OR sponsor LIKE '%abbvie%' OR sponsor LIKE '%boehringer%'
       OR sponsor LIKE '%janssen%' OR sponsor LIKE '%johnson%' OR sponsor LIKE '%pfizer%'
       OR sponsor LIKE '%gilead%')
GROUP BY sponsor
ORDER BY sponsor
"""
print(pd.read_sql(q, conn).to_string())

                                                                                      sponsor  studies
0                                                                                      AbbVie       37
1                                                                    Ablynx, a Sanofi company        3
2                                                                                       Amgen       44
3                                                                                 AstraZeneca       59
4                                                                        Boehringer Ingelheim       28
5                                                                        Bristol-Myers Squibb       93
6        Cidara Therapeutics Inc., a subsidiary of Merck & Co., Inc. (Rahway, New Jersey USA)        1
7                                       Dicerna Pharmaceuticals, Inc., a Novo Nordisk company        1
8                                                                       E

In [44]:
#top 15 sponsors
conn = sqlite3.connect("../data/clean/trials.db")

q7 = """
SELECT COALESCE(a.sponsorGroup, s.sponsor) AS sponsorName,
       COUNT(*) AS studies,
       RANK() OVER (ORDER BY COUNT(*) DESC) AS rnk,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM studies), 1) AS pctOfStudies
FROM studies AS s
LEFT JOIN sponsor_alias AS a ON a.sponsor = s.sponsor
WHERE s.sponsorClass = 'INDUSTRY'
GROUP BY sponsorName
ORDER BY studies DESC
LIMIT 15
"""
print(pd.read_sql(q7, conn))

                 sponsorName  studies  rnk  pctOfStudies
0       Bristol-Myers Squibb      100    1           5.8
1                  Eli Lilly       66    2           3.8
2          Merck & Co. (MSD)       65    3           3.8
3   Novartis Pharmaceuticals       60    4           3.5
4                AstraZeneca       59    5           3.4
5                     Sanofi       46    6           2.7
6                      Roche       46    6           2.7
7                        GSK       44    8           2.6
8                      Amgen       44    8           2.6
9               Novo Nordisk       38   10           2.2
10                    AbbVie       37   11           2.2
11      Boehringer Ingelheim       28   12           1.6
12         Johnson & Johnson       27   13           1.6
13                    Pfizer       23   14           1.3
14           Gilead Sciences       23   14           1.3


In [46]:
#industry vs academic sponsors
q8 = """
SELECT CASE WHEN sponsorClass = 'INDUSTRY' THEN 'Industry'
            ELSE 'Academic and other' END AS sponsorType,
       COUNT(*) AS studies,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM studies), 1) AS pct
FROM studies
GROUP BY sponsorType
"""
print(pd.read_sql(q8, conn))

          sponsorType  studies   pct
0  Academic and other      284  16.5
1            Industry     1433  83.5
